In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numba import njit
import seaborn as sns
from src.generative_models import sample_pt_model, sample_mvl_model
from src.context import get_context
from src.priors import sample_pt_prior
import bayesflow as bf

In [2]:
context_gen = bf.simulation.ContextGenerator(
    batchable_context_fun=get_context
)

In [3]:
def configurator(forward_dict):
    out_dict = {}
    data = forward_dict["sim_data"][:, :, None]
    context = np.array(forward_dict["sim_batchable_context"]) / 200
    out_dict["summary_conditions"] = np.c_[data, context].astype(np.float32)
    out_dict["parameters"] = forward_dict["prior_draws"].astype(np.float32)
    return out_dict

## PT Model

In [4]:
param_names = (r'$\lambda$', r'$\alpha$', r'$\tau$')

In [5]:
prior = bf.simulation.Prior(
    batch_prior_fun=sample_pt_prior,
    param_names=param_names
)

In [ ]:
simulator = bf.simulation.Simulator(
    simulator_fun=sample_pt_model,
    context_generator=context_gen
)

model = bf.simulation.GenerativeModel(
    prior=prior,
    simulator=simulator,
    name="pt_model"
)

In [ ]:
%%time
_ = model(128)

In [13]:
summary_net = bf.networks.SetTransformer(input_dim=7, summary_dim=32)

inference_net = bf.networks.InvertibleNetwork(
    num_params=len(prior.param_names),
    coupling_settings={"dense_args": dict(kernel_regularizer=None), "dropout": False},
)

In [ ]:
amortizer = bf.amortizers.AmortizedPosterior(inference_net, summary_net)

trainer = bf.trainers.Trainer(
    generative_model=model, 
    amortizer=amortizer, 
    configurator=configurator, 
    checkpoint_path=f"checkpoints/{model.name}",
    max_to_keep=1
)

In [ ]:
history = trainer.train_online(100, 1000, 128)